# GPT-2 Small: Accessibility Domain Knowledge

12 layers | 12 heads | d_model=768 | d_mlp=3072 | sequential attn+MLP

In [2]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [3]:
model_name = "gpt2-small"
model, info = load_model(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Loaded gpt2-small on mps
  12 layers | 12 heads | d_model=768 | d_mlp=3072 | sequential attn+MLP


In [4]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: gpt2
Prompt: "The W in WCAG stands for"
Target: " Web" (token 5313)
Final prediction: " ""

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         the                 3312           0.000002    
1         the                 4146           0.000001    
2         the                 4539           0.000000    
3         the                 2807           0.000001    
4         the                 3310           0.000001    
5         the                 6119           0.000001    
6         "                   4441           0.000001    
7         "                   5226           0.000000    
8         Water               206            0.000162    
9         Water               45             0.002258    
10        World               45             0.002519    
11        "                   20             0.004592    
Model: gpt2
Target: " Web"

Layer    Attn→Targ

In [5]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [6]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/gpt2/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/gpt2/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/gpt2/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/gpt2/{model_name}/figures")

Saved: ../../results/gpt2/gpt2-small/figures/wcag-logit-lens.png
Saved: ../../results/gpt2/gpt2-small/figures/wcag-decomposition.png
Saved: ../../results/gpt2/gpt2-small/figures/wcag-head-heatmap.png
Saved: ../../results/gpt2/gpt2-small/figures/aria-logit-lens.png
Saved: ../../results/gpt2/gpt2-small/figures/aria-decomposition.png
Saved: ../../results/gpt2/gpt2-small/figures/aria-head-heatmap.png
Saved: ../../results/gpt2/gpt2-small/figures/alt-logit-lens.png
Saved: ../../results/gpt2/gpt2-small/figures/alt-decomposition.png
Saved: ../../results/gpt2/gpt2-small/figures/alt-head-heatmap.png
Saved: ../../results/gpt2/gpt2-small/figures/HTML-logit-lens.png
Saved: ../../results/gpt2/gpt2-small/figures/HTML-decomposition.png
Saved: ../../results/gpt2/gpt2-small/figures/HTML-head-heatmap.png
Saved: ../../results/gpt2/gpt2-small/figures/screenReader-logit-lens.png
Saved: ../../results/gpt2/gpt2-small/figures/screenReader-decomposition.png
Saved: ../../results/gpt2/gpt2-small/figures/screenRea

In [7]:
unload(model)

Model unloaded, memory cleared
